In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install transformers wandb -q

# Model 2 (pretrained BERT)

The idea is to format each question+option pair as:
* [CLS] question [SEP] option [SEP]
* and fine-tune BERT to classify which option is correct.

In [ ]:
import os
os.environ["WANDB_API_KEY"] = "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"  # storing key

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
import wandb

# Load Data 
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

# Config 
MODEL_NAME = 'bert-base-uncased'
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5

wandb.init(project="24f3002284-t22026", name="model2-bert", config={
    "model": MODEL_NAME,
    "max_len": MAX_LEN,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "lr": LR
})

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

# Dataset
class MCQDataset(Dataset):
    def __init__(self, df, is_test=False):
        self.df = df.reset_index(drop=True)
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        encodings = []
        for opt in ['A', 'B', 'C', 'D', 'E']:
            enc = tokenizer(
                str(row['prompt']),
                str(row[opt]),
                max_length=MAX_LEN,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )
            encodings.append({
                'input_ids': enc['input_ids'].squeeze(),
                'attention_mask': enc['attention_mask'].squeeze()
            })
        if not self.is_test:
            label = torch.tensor(label_map[row['answer']], dtype=torch.long)
            return encodings, label
        return encodings

# Model 
class BERTMCQModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        self.classifier = nn.Linear(self.bert.config.hidden_size, 1)

    def forward(self, encodings):
        logits = []
        for enc in encodings:
            output = self.bert(
                input_ids=enc['input_ids'],
                attention_mask=enc['attention_mask']
            )
            cls = output.last_hidden_state[:, 0, :]
            logits.append(self.classifier(cls))
        return torch.cat(logits, dim=1)  # (batch, 5)

# Collate 
def collate_fn(batch):
    if len(batch[0]) == 2:
        encodings_list, labels = zip(*batch)
        labels = torch.stack(labels)
    else:
        encodings_list = batch
        labels = None

    batch_encodings = []
    for i in range(5):
        input_ids = torch.stack([e[i]['input_ids'] for e in encodings_list])
        attention_mask = torch.stack([e[i]['attention_mask'] for e in encodings_list])
        batch_encodings.append({'input_ids': input_ids, 'attention_mask': attention_mask})
    return batch_encodings, labels

# Training 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

train_df, val_df = train_test_split(train, test_size=0.2, random_state=42)
train_loader = DataLoader(MCQDataset(train_df), batch_size=BATCH_SIZE,
                         shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(MCQDataset(val_df), batch_size=BATCH_SIZE,
                       collate_fn=collate_fn)

model = BERTMCQModel().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    model.train()
    total_loss, correct = 0, 0
    for encodings, labels in train_loader:
        encodings = [{k: v.to(device) for k, v in e.items()} for e in encodings]
        labels = labels.to(device)
        optimizer.zero_grad()
        logits = model(encodings)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (logits.argmax(1) == labels).sum().item()

    train_acc = correct / len(train_df)

    model.eval()
    val_correct = 0
    with torch.no_grad():
        for encodings, labels in val_loader:
            encodings = [{k: v.to(device) for k, v in e.items()} for e in encodings]
            labels = labels.to(device)
            logits = model(encodings)
            val_correct += (logits.argmax(1) == labels).sum().item()
    val_acc = val_correct / len(val_df)

    print(f"Epoch {epoch+1}: Loss={total_loss/len(train_loader):.4f}, Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}")
    wandb.log({"epoch": epoch+1, "loss": total_loss/len(train_loader),
               "train_acc": train_acc, "val_acc": val_acc})

torch.save(model.state_dict(), 'model2_bert.pth')
wandb.save('model2_bert.pth')
wandb.finish()
print("Training done!")

In [ ]:
# Inference with BERT Model
model.eval()
test_dataset = MCQDataset(test, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn)

all_preds = []
with torch.no_grad():
    for encodings, _ in test_loader:
        encodings = [{k: v.to(device) for k, v in e.items()} for e in encodings]
        logits = model(encodings)
        top3 = torch.topk(logits, 3, dim=1).indices.cpu().numpy()
        all_preds.extend(top3)

options = ['A', 'B', 'C', 'D', 'E']
predictions = [' '.join([options[i] for i in pred]) for pred in all_preds]

submission = pd.DataFrame({'ID': test['id'], 'Prediction': predictions})
submission.to_csv('submission.csv', index=False)
print(submission.head())
print("Done!")